In [0]:
print("hello")

print("\n" + "=" * 80)

In [0]:
tables = [
    "samples.tpch.lineitem",
    "samples.tpch.nation",
    "samples.tpch.orders",
    "samples.tpch.part",
    "samples.tpch.partsupp",
    "samples.tpch.region",
    "samples.tpch.supplier",
    "samples.tpch.customer",
]

print("=" * 80)
print("TABLE SCHEMAS")
print("=" * 80)

for t in tables:
    df = spark.table(t)
    print(f"\n--- {t} ---")
    df.printSchema()
    print(f"Row count: {df.count():,}")

print("\n" + "=" * 80)
print("TABLE RELATIONSHIPS (TPCH Schema)")
print("=" * 80)
print("""
1. region  ──<  nation        (region.r_regionkey = nation.n_regionkey)
2. nation  ──<  customer       (nation.n_nationkey = customer.c_nationkey)
3. nation  ──<  supplier       (nation.n_nationkey = supplier.s_nationkey)
4. customer ──< orders         (customer.c_custkey = orders.o_custkey)
5. orders  ──< lineitem       (orders.o_orderkey = lineitem.l_orderkey)
6. supplier ──< partsupp      (supplier.s_suppkey = partsupp.ps_suppkey)
7. part    ──< partsupp      (part.p_partkey = partsupp.ps_partkey)
8. partsupp ──< lineitem      (partsupp.ps_partkey = lineitem.l_partkey
                               AND partsupp.ps_suppkey = lineitem.l_suppkey)
9. supplier ──< lineitem     (supplier.s_suppkey = lineitem.l_suppkey)
10. part   ──< lineitem      (part.p_partkey = lineitem.l_partkey)

Entity overview:
  - region:    Geographic regions (top-level)
  - nation:    Countries, each belongs to a region
  - customer:  Customers, each located in a nation
  - supplier:  Suppliers, each located in a nation
  - part:      Parts (products)
  - partsupp:  Supplier-part mapping (which supplier supplies which part)
  - orders:    Customer orders
  - lineitem: Individual line items within orders, referencing parts and suppliers
""")

In [0]:
tables = [
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
    "customer",
]

print("=" * 80)
print("ROW COUNT COMPARISON: samples.tpch vs tpch.bronze")
print("=" * 80)
print(f"{'Table':<15} {'samples.tpch':>20} {'tpch.bronze':>20} {'Match':>10}")
print("-" * 80)

for t in tables:
    sample_table = f"samples.tpch.{t}"
    bronze_table = f"tpch.bronze.{t}"

    sample_count = spark.table(sample_table).count()
    try:
        bronze_count = spark.table(bronze_table).count()
    except Exception as e:
        bronze_count = None
        bronze_err = str(e)

    if bronze_count is None:
        print(f"{t:<15} {sample_count:>20,} {'N/A':>20} {'ERROR':>10}")
        print(f"  └── Error reading {bronze_table}: {bronze_err}")
    else:
        match = "✅ Yes" if sample_count == bronze_count else "❌ No"
        diff = sample_count - bronze_count
        diff_str = f" (diff: {diff:+,})" if diff != 0 else ""
        print(f"{t:<15} {sample_count:>20,} {bronze_count:>20,} {match:>10}{diff_str}")

print("=" * 80)

In [0]:
tables = [
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
    "customer",
]

print("=" * 80)
print("ROW COUNTS: tpch.silver")
print("=" * 80)
print(f"{'Table':<15} {'Row Count':>20}")
print("-" * 80)

for t in tables:
    silver_table = f"tpch.silver.{t}"
    try:
        count = spark.table(silver_table).count()
        print(f"{t:<15} {count:>20,}")
    except Exception as e:
        print(f"{t:<15} {'N/A':>20}")
        print(f"  └── Error reading {silver_table}: {e}")

print("=" * 80)

In [0]:
tables = [
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
    "customer",
]

print("=" * 80)
print("DROPPING ALL TABLES IN tpch.bronze")
print("=" * 80)

for t in tables:
    bronze_table = f"tpch.bronze.{t}"
    try:
        spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
        print(f"  ✅ Dropped {bronze_table}")
    except Exception as e:
        print(f"  ❌ Failed to drop {bronze_table}: {e}")

print("\n" + "=" * 80)